# Football Win Probability Model
**Any two international / club teams | football-data.org API**

Approach: multi-feature logistic regression with **competition importance weighting**
(World Cup / Euro / Nations League matches weighted higher than friendlies),
squad depth scoring pulled from the teams endpoint, and calibration via Platt scaling.

> Register at https://www.football-data.org for your free API token.

**Quick start:** fill in `API_TOKEN`, `TEAM_A_NAME`, `TEAM_B_NAME`, and `MATCH_DATE` in the Config cell below — everything else is automatic.

In [ ]:
# pip install requests pandas scikit-learn matplotlib numpy
import requests
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── CONFIG — edit these four lines ────────────────────────────────────────────
API_TOKEN    = "YOUR_API_TOKEN_HERE"   # <- paste your token

TEAM_A_NAME  = "France"                # <- Team A (displayed first / treated as 'home' perspective)
TEAM_B_NAME  = "Senegal"               # <- Team B
MATCH_DATE   = "2026-06-16"            # <- YYYY-MM-DD of the match you're predicting

# Optional: override team IDs directly (set to None to use name search)
TEAM_A_ID    = None   # e.g. 773 for France
TEAM_B_ID    = None   # e.g. 1991 for Senegal

# How far back to pull historical matches
DATE_FROM    = "2019-01-01"
# ──────────────────────────────────────────────────────────────────────────────

BASE_URL = "https://api.football-data.org/v4"
HEADERS  = {"X-Auth-Token": API_TOKEN}

# Competition importance multipliers
COMP_WEIGHTS = {
    "WC":  3.0,
    "EC":  2.5,   # UEFA Euros
    "CAN": 2.5,   # AFCON
    "UCL": 2.0,
    "UNL": 1.8,   # Nations League
    "WCQ": 1.5,   # WC Qualifying
    "FR":  0.6,   # Friendlies
}
DEFAULT_COMP_WEIGHT = 1.0

# Per-team colour palette (fallbacks are applied for unknown teams)
TEAM_COLORS = {
    "France":      "#002395",
    "Senegal":     "#00853F",
    "Brazil":      "#009C3B",
    "Argentina":   "#74ACDF",
    "Germany":     "#000000",
    "Spain":       "#AA151B",
    "England":     "#CF081F",
    "Portugal":    "#006600",
    "Netherlands": "#FF6600",
    "Italy":       "#003399",
    "Belgium":     "#EF3340",
    "Uruguay":     "#5AAFF6",
    "Croatia":     "#FF0000",
    "Morocco":     "#C1272D",
    "Japan":       "#BC002D",
    "USA":         "#B31942",
    "Mexico":      "#006847",
    "Colombia":    "#FCD116",
    "Ecuador":     "#FFD100",
    "Australia":   "#00843D",
    "Switzerland": "#FF0000",
    "Denmark":     "#C60C30",
    "Poland":      "#DC143C",
    "Serbia":      "#C6363C",
    "Ghana":       "#006B3F",
    "Cameroon":    "#007A5E",
    "Nigeria":     "#008751",
    "South Korea": "#003478",
    "Iran":        "#239F40",
    "Saudi Arabia":"#006C35",
    "Qatar":       "#8D1B3D",
    "Tunisia":     "#E70013",
    "Costa Rica":  "#002B7F",
    "Canada":      "#FF0000",
    "Wales":       "#CF101A",
    "Czech Republic": "#D7141A",
    "Hungary":     "#CE2939",
    "Scotland":    "#003DA5",
    "Turkey":      "#E30A17",
    "Ukraine":     "#005BBB",
    "Austria":     "#ED2939",
    "Sweden":      "#006AA7",
    "Norway":      "#EF2B2D",
    "Romania":     "#002B7F",
    "Slovakia":    "#005B96",
    "Slovenia":    "#003DA5",
    "Greece":      "#0D5EAF",
    "Albania":     "#E41E20",
    "Georgia":     "#FF0000",
    "Iceland":     "#003897",
    "Finland":     "#003580",
    "Bolivia":     "#F4E400",
    "Peru":        "#D91023",
    "Chile":       "#D52B1E",
    "Venezuela":   "#CF142B",
    "Paraguay":    "#D52B1E",
    "Honduras":    "#0073CF",
    "Jamaica":     "#000000",
    "El Salvador": "#0F47AF",
    "Panama":      "#DA121A",
    "Guatemala":   "#4997D0",
    "Haiti":       "#00209F",
    "Cuba":        "#002A8F",
    "Egypt":       "#CE1126",
    "Algeria":     "#006233",
    "South Africa":"#007A4D",
    "Ivory Coast": "#F77F00",
    "Mali":        "#009A00",
    "Guinea":      "#CE1126",
    "DR Congo":    "#007FFF",
    "Zambia":      "#198A00",
    "Angola":      "#CC0000",
    "Kenya":       "#006600",
    "Tanzania":    "#1EB53A",
    "Ethiopia":    "#078930",
    "Indonesia":   "#CE1126",
    "Thailand":    "#A51931",
    "Vietnam":     "#DA251D",
    "China":       "#DE2910",
    "India":       "#FF9933",
    "Iraq":        "#CE1126",
    "Jordan":      "#007A3D",
    "Bahrain":     "#CE1126",
    "UAE":         "#00732F",
    "Russia":      "#D52B1E",
    "New Zealand": "#00247D",
}
FALLBACK_COLORS = ["#1f77b4", "#ff7f0e"]

def team_color(name, idx):
    return TEAM_COLORS.get(name, FALLBACK_COLORS[idx % 2])

print(f"Config loaded — predicting: {TEAM_A_NAME} vs {TEAM_B_NAME} on {MATCH_DATE}")

In [ ]:
# ── team lookup ───────────────────────────────────────────────────────────────
def search_team_id(name):
    """Return the first matching team ID from football-data.org."""
    url = f"{BASE_URL}/teams"
    r = requests.get(url, headers=HEADERS, params={"name": name})
    if r.status_code != 200:
        raise ValueError(f"Team search failed (HTTP {r.status_code}) for '{name}'. "
                         "Try setting TEAM_A_ID / TEAM_B_ID manually.")
    data = r.json()
    teams = data.get("teams", [])
    if not teams:
        # Some tiers of the free API don't support /teams search — guide the user
        raise ValueError(
            f"No team found for '{name}'. "
            "Visit https://www.football-data.org/documentation/quickstart to look up the numeric ID, "
            "then set TEAM_A_ID / TEAM_B_ID directly."
        )
    team = teams[0]
    print(f"  Found: {team['name']} (ID {team['id']})")
    return team["id"]

TEAM_IDS = {}
for team_name, override_id in [(TEAM_A_NAME, TEAM_A_ID), (TEAM_B_NAME, TEAM_B_ID)]:
    if override_id is not None:
        TEAM_IDS[team_name] = override_id
        print(f"{team_name}: using provided ID {override_id}")
    else:
        print(f"Searching for team: {team_name}")
        TEAM_IDS[team_name] = search_team_id(team_name)

print("\nTeam IDs:", TEAM_IDS)

In [ ]:
# ── fetch matches & squad sizes ────────────────────────────────────────────────
def fetch_matches(team_id, date_from=DATE_FROM):
    url    = f"{BASE_URL}/teams/{team_id}/matches"
    params = {"dateFrom": date_from, "status": "FINISHED", "limit": 100}
    r = requests.get(url, headers=HEADERS, params=params)
    r.raise_for_status()
    return r.json().get("matches", [])

def fetch_squad_size(team_id):
    url = f"{BASE_URL}/teams/{team_id}"
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    return len(r.json().get("squad", []))

raw         = {}
squad_sizes = {}
for name, tid in TEAM_IDS.items():
    raw[name] = fetch_matches(tid)
    print(f"{name}: {len(raw[name])} matches fetched")
    try:
        squad_sizes[name] = fetch_squad_size(tid)
        print(f"{name}: squad size = {squad_sizes[name]}")
    except Exception as e:
        squad_sizes[name] = 23
        print(f"  squad fetch failed ({e}), using default 23")

In [ ]:
# ── parse with competition weighting & recency decay ──────────────────────────
def parse_matches(matches, team_id, comp_weights=COMP_WEIGHTS):
    rows = []
    for m in matches:
        ft = m.get("score", {}).get("fullTime", {})
        if ft.get("home") is None:
            continue

        is_home = m["homeTeam"]["id"] == team_id
        gf = ft["home"] if is_home else ft["away"]
        ga = ft["away"] if is_home else ft["home"]

        comp_code = m.get("competition", {}).get("code", "UNK")
        comp_w    = comp_weights.get(comp_code, DEFAULT_COMP_WEIGHT)

        # recency decay: half-life 365 days from match day
        date      = pd.to_datetime(m["utcDate"][:10])
        days_back = (pd.Timestamp(MATCH_DATE) - date).days
        time_w    = 2 ** (-days_back / 365)

        rows.append({
            "date":    date,
            "comp":    comp_code,
            "is_home": int(is_home),
            "gf":      gf,
            "ga":      ga,
            "gd":      gf - ga,
            "win":     int(gf > ga),
            "draw":    int(gf == ga),
            "points":  3 if gf > ga else (1 if gf == ga else 0),
            "comp_w":  comp_w,
            "time_w":  time_w,
            "weight":  comp_w * time_w,
        })

    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    return df

dfs = {n: parse_matches(raw[n], tid) for n, tid in TEAM_IDS.items()}

for name, df in dfs.items():
    print(f"\n{name}: {len(df)} matches, competition distribution:")
    print(df["comp"].value_counts().to_string())

In [ ]:
# ── weighted aggregate stats per team ─────────────────────────────────────────
def weighted_stats(df):
    w    = df["weight"].values
    wsum = max(w.sum(), 1e-9)
    return {
        "win_rate":     np.dot(df["win"],    w) / wsum,
        "draw_rate":    np.dot(df["draw"],   w) / wsum,
        "gf_per_game":  np.dot(df["gf"],     w) / wsum,
        "ga_per_game":  np.dot(df["ga"],     w) / wsum,
        "gd_per_game":  np.dot(df["gd"],     w) / wsum,
        "pts_per_game": np.dot(df["points"], w) / wsum,
        "home_rate":    df["is_home"].mean(),
    }

stats = {n: weighted_stats(dfs[n]) for n in TEAM_IDS}

print("\nWeighted aggregate stats:")
print(pd.DataFrame(stats).T.round(3).to_string())

In [ ]:
# ── build training dataset ─────────────────────────────────────────────────────
# Slide a window of WINDOW matches; build differential feature vectors.

WINDOW = 12

def windowed_features(df, window=WINDOW):
    out = []
    for i in range(window, len(df)):
        chunk = df.iloc[i-window:i]
        w     = chunk["weight"].values
        ws    = max(w.sum(), 1e-9)
        out.append({
            "win_rate":   np.dot(chunk["win"],    w) / ws,
            "draw_rate":  np.dot(chunk["draw"],   w) / ws,
            "gf_rate":    np.dot(chunk["gf"],     w) / ws,
            "ga_rate":    np.dot(chunk["ga"],     w) / ws,
            "gd_rate":    np.dot(chunk["gd"],     w) / ws,
            "pts_rate":   np.dot(chunk["points"], w) / ws,
            "date":       df.iloc[i]["date"],
            "actual_win": df.iloc[i]["win"],
            "is_home":    df.iloc[i]["is_home"],
        })
    return pd.DataFrame(out)

wf = {n: windowed_features(dfs[n]) for n in TEAM_IDS}

FCOLS = ["win_rate","draw_rate","gf_rate","ga_rate","gd_rate","pts_rate"]

def build_diff_dataset(wf1, wf2, sq1, sq2):
    n    = min(len(wf1), len(wf2))
    a, b = wf1.tail(n).reset_index(drop=True), wf2.tail(n).reset_index(drop=True)
    diff = a[FCOLS].values - b[FCOLS].values
    out  = pd.DataFrame(diff, columns=[f"d_{c}" for c in FCOLS])
    out["home_adv"]   = a["is_home"].values
    out["squad_diff"] = (sq1 - sq2) / max(sq1, sq2, 1)
    out["label"]      = a["actual_win"].values
    return out

team_a, team_b = list(TEAM_IDS.keys())
train = build_diff_dataset(
    wf[team_a], wf[team_b],
    squad_sizes[team_a], squad_sizes[team_b]
)
print(f"Training rows : {len(train)}")
print(f"Label balance : {train['label'].mean():.2f} (fraction wins for {team_a})")

In [ ]:
# ── logistic regression + Platt calibration ────────────────────────────────────
FEATS = [c for c in train.columns if c != "label"]
X = train[FEATS].values
y = train["label"].values

base_clf = LogisticRegression(C=1.5, solver="lbfgs", max_iter=500, random_state=99)
model = CalibratedClassifierCV(
    estimator=Pipeline([("sc", StandardScaler()), ("clf", base_clf)]),
    cv=5,
    method="sigmoid"
)

n_splits = min(5, int(y.sum()), int((1 - y).sum()))
if n_splits < 2:
    print("WARNING: not enough class diversity for cross-validation — fitting on full data.")
    model.fit(X, y)
    print("Model fitted (no CV scores available).")
else:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=99)
    raw_auc = cross_val_score(
        Pipeline([("sc", StandardScaler()), ("clf", base_clf)]),
        X, y, cv=cv, scoring="roc_auc"
    )
    brier = cross_val_score(model, X, y, cv=cv, scoring="neg_brier_score")
    print(f"CV ROC-AUC : {raw_auc.mean():.3f} ± {raw_auc.std():.3f}")
    print(f"CV Brier   : {-brier.mean():.3f} ± {brier.std():.3f}  (lower = better)")
    model.fit(X, y)

In [ ]:
# ── prediction for MATCH_DATE ─────────────────────────────────────────────────
a_f = wf[team_a].iloc[-1]
b_f = wf[team_b].iloc[-1]

sq_max = max(squad_sizes.values())
pred_vec = np.array([[a_f[c] - b_f[c] for c in FCOLS] +
                     [0,   # neutral venue
                      (squad_sizes[team_a] - squad_sizes[team_b]) / sq_max]])

probs    = model.predict_proba(pred_vec)[0]
a_win    = probs[1]
b_win    = probs[0]
implied_draw = max(0.0, 1 - a_win - b_win)

print("=" * 55)
print(f"  {team_a} vs {team_b} — {MATCH_DATE}")
print("=" * 55)
print(f"  {team_a:<20} win : {a_win:.1%}")
print(f"  {team_b:<20} win : {b_win:.1%}")
print(f"  Implied draw           : ~{implied_draw:.1%}")
print("=" * 55)

In [ ]:
# ── visualizations ────────────────────────────────────────────────────────────
col_a = team_color(team_a, 0)
col_b = team_color(team_b, 1)

fig = plt.figure(figsize=(14, 10))
fig.suptitle(f"{team_a} vs {team_b} — {MATCH_DATE}",
             fontsize=13, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

# 1. Win-probability donut
ax1 = fig.add_subplot(gs[0, 0])
wedges, _ = ax1.pie(
    [a_win, b_win],
    colors=[col_a, col_b],
    startangle=90,
    wedgeprops={"width": 0.55, "edgecolor": "white"}
)
ax1.legend(wedges,
           [f"{team_a}\n{a_win:.1%}", f"{team_b}\n{b_win:.1%}"],
           loc="lower center", bbox_to_anchor=(0.5, -0.15), fontsize=9)
ax1.set_title("Win Probability (calibrated LR)")

# 2. Competition mix comparison
ax2 = fig.add_subplot(gs[0, 1])
top_comps = list(COMP_WEIGHTS.keys())
a_counts  = dfs[team_a]["comp"].value_counts().reindex(top_comps, fill_value=0)
b_counts  = dfs[team_b]["comp"].value_counts().reindex(top_comps, fill_value=0)
x_c = np.arange(len(top_comps))
ax2.bar(x_c - 0.2, a_counts.values, width=0.4, label=team_a, color=col_a, alpha=0.85)
ax2.bar(x_c + 0.2, b_counts.values, width=0.4, label=team_b, color=col_b, alpha=0.85)
ax2.set_xticks(x_c); ax2.set_xticklabels(top_comps, rotation=45, ha="right")
ax2.set_title("Match Count by Competition")
ax2.legend(); ax2.spines[["top","right"]].set_visible(False)

# 3. Aggregate stats comparison
ax3 = fig.add_subplot(gs[0, 2])
cats3 = ["GF/game", "GA/game", "Win%", "Pts/game"]
keys3 = ["gf_per_game", "ga_per_game", "win_rate", "pts_per_game"]
a3    = [stats[team_a][k] for k in keys3]
b3    = [stats[team_b][k] for k in keys3]
x3    = np.arange(len(cats3))
ax3.bar(x3 - 0.2, a3, width=0.4, label=team_a, color=col_a, alpha=0.85)
ax3.bar(x3 + 0.2, b3, width=0.4, label=team_b, color=col_b, alpha=0.85)
ax3.set_xticks(x3); ax3.set_xticklabels(cats3, rotation=20, ha="right")
ax3.set_title("Weighted Aggregate Stats")
ax3.legend(); ax3.spines[["top","right"]].set_visible(False)

# 4. Rolling form over time
ax4 = fig.add_subplot(gs[1, :2])
ax4.plot(wf[team_a]["date"], wf[team_a]["win_rate"], label=team_a, color=col_a, lw=2)
ax4.plot(wf[team_b]["date"], wf[team_b]["win_rate"], label=team_b, color=col_b, lw=2)
ax4.set_title(f"Windowed Win Rate (window={WINDOW}, competition-weighted)")
ax4.set_ylabel("Win Rate")
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax4.legend(); ax4.grid(axis="y", alpha=0.3)
ax4.spines[["top","right"]].set_visible(False)

# 5. Goals trend
ax5 = fig.add_subplot(gs[1, 2])
ax5.plot(wf[team_a]["date"], wf[team_a]["gf_rate"], color=col_a, lw=2,   label=f"{team_a} GF")
ax5.plot(wf[team_a]["date"], wf[team_a]["ga_rate"], color=col_a, lw=1.5, linestyle="--", label=f"{team_a} GA")
ax5.plot(wf[team_b]["date"], wf[team_b]["gf_rate"], color=col_b, lw=2,   label=f"{team_b} GF")
ax5.plot(wf[team_b]["date"], wf[team_b]["ga_rate"], color=col_b, lw=1.5, linestyle="--", label=f"{team_b} GA")
ax5.set_title("Weighted GF / GA per Game")
ax5.legend(fontsize=7); ax5.grid(axis="y", alpha=0.3)
ax5.spines[["top","right"]].set_visible(False)

out_img = f"{team_a.replace(' ','_')}_vs_{team_b.replace(' ','_')}_output.png"
plt.savefig(out_img, dpi=130, bbox_inches="tight")
plt.show()
print(f"saved → {out_img}")

In [ ]:
# ── sensitivity analysis: WC weight multiplier ────────────────────────────────
results = []
for wc_boost in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    local_weights = COMP_WEIGHTS.copy()
    local_weights["WC"] = wc_boost

    dfs_l = {n: parse_matches(raw[n], tid, comp_weights=local_weights)
             for n, tid in TEAM_IDS.items()}
    wf_l  = {n: windowed_features(dfs_l[n]) for n in TEAM_IDS}
    tr_l  = build_diff_dataset(wf_l[team_a], wf_l[team_b],
                               squad_sizes[team_a], squad_sizes[team_b])
    Xl, yl = tr_l[FEATS].values, tr_l["label"].values
    if len(np.unique(yl)) < 2:
        continue
    n_cv = min(5, int(yl.sum()), int((1 - yl).sum()))
    if n_cv < 2:
        continue
    m_l = CalibratedClassifierCV(
        Pipeline([("sc", StandardScaler()),
                  ("clf", LogisticRegression(C=1.5, max_iter=500))]),
        cv=n_cv, method="sigmoid"
    )
    m_l.fit(Xl, yl)
    af = wf_l[team_a].iloc[-1]
    bf = wf_l[team_b].iloc[-1]
    pv = np.array([[af[c] - bf[c] for c in FCOLS] + [0, pred_vec[0, -1]]])
    p  = m_l.predict_proba(pv)[0]
    results.append({"WC_weight": wc_boost,
                    f"{team_a}_win": p[1],
                    f"{team_b}_win": p[0]})

sens_df = pd.DataFrame(results)
print("\nSensitivity to World Cup match weight:")
print(sens_df.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sens_df["WC_weight"], sens_df[f"{team_a}_win"], "o-", color=col_a, lw=2, label=team_a)
ax.plot(sens_df["WC_weight"], sens_df[f"{team_b}_win"], "o-", color=col_b, lw=2, label=team_b)
ax.set_xlabel("WC competition weight multiplier")
ax.set_ylabel("Win Probability")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_title(f"Sensitivity: WC weight vs Win Probability ({team_a} vs {team_b})")
ax.legend(); ax.grid(alpha=0.3)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
sens_img = f"{team_a.replace(' ','_')}_vs_{team_b.replace(' ','_')}_sensitivity.png"
plt.savefig(sens_img, dpi=130, bbox_inches="tight")
plt.show()
print(f"saved → {sens_img}")